In [16]:
import os
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Set padding token for the tokenizer
tokenizer.pad_token = tokenizer.eos_token  # Use '' as the padding token

# Specify the path to the folder containing your text documents on Google Drive
data_folder = '/content/drive/MyDrive/Object_statutes'

# Check if the specified directory exists
if os.path.exists(data_folder):
    # List all files in the directory with .txt extension
    files = os.listdir(data_folder)
    text_files = [file for file in files if file.endswith('.txt')]
    print(f"Found {len(text_files)} text files in the directory.")

    # Load and process text data from each file
    texts = []
    for file in text_files:
        file_path = os.path.join(data_folder, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
            texts.append(text)

    print("Text documents loaded successfully.")

    # Tokenize the text data using GPT-2 tokenizer with padding
    inputs = tokenizer('\n\n'.join(texts), return_tensors='pt', max_length=1024, truncation=True, padding=True)
    print("Text data tokenized.")

    # Fine-tune the GPT-2 model on your data
    model.train()
    model.resize_token_embeddings(len(tokenizer))
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

    for epoch in range(3):  # Adjust number of epochs as needed
        outputs = model(**inputs, labels=inputs['input_ids'])
        loss = outputs.loss
        print(f"Epoch {epoch + 1} Loss: {loss.item()}")

        # Backpropagation and optimization
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print("Model training completed.")

    # Save the fine-tuned model to a specified directory
    output_dir = '/content/drive/MyDrive/saved_gpt2_model'
    os.makedirs(output_dir, exist_ok=True)  # Create output directory if it doesn't exist
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Fine-tuned model saved to: {output_dir}")

else:
    print(f"Directory '{data_folder}' does not exist.")

Found 10 text files in the directory.
Text documents loaded successfully.
Text data tokenized.
Epoch 1 Loss: 1.9324980974197388
Epoch 2 Loss: 1.7166882753372192
Epoch 3 Loss: 1.6310628652572632
Model training completed.
Fine-tuned model saved to: /content/drive/MyDrive/saved_gpt2_model


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Specify the path to the directory containing the saved model
saved_model_dir = '/content/drive/MyDrive/saved_gpt2_model'

# Load the saved GPT-2 model and tokenizer
loaded_tokenizer = GPT2Tokenizer.from_pretrained(saved_model_dir)
loaded_model = GPT2LMHeadModel.from_pretrained(saved_model_dir)

# Example usage: Generate text using the loaded model
prompt = "corruption"
input_ids = loaded_tokenizer.encode(prompt, return_tensors='pt')
generated_sequence = loaded_model.generate(input_ids, max_length=100, num_return_sequences=1, temperature=0.7)
decoded_sequence = loaded_tokenizer.decode(generated_sequence[0], skip_special_tokens=True)

print("Generated Sequence:")
print(decoded_sequence)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Sequence:
corruption, and the threat of violence.

The government has also been accused of failing to protect the rights of women and girls, and of failing to protect the rights of the LGBT community.

The government has also been accused of failing to protect the rights of women and girls, and of failing to protect the rights of the LGBT community.

The government has also been accused of failing to protect the rights of women and girls, and of failing to protect the rights of the LGBT community
